# Level 0 single-seed diagnostics

Primary metrics are held-out loss and perplexity. Exact GPT-2 BPE-token accuracy is secondary.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getenv('NANOGPT_LEVEL0_RESULTS_ROOT', '/tmp/nanogpt-level0-bpe/results'))
OPT = os.getenv('NANOGPT_LEVEL0_NOTEBOOK_OPTIMIZER', 'adamw')
SEED = int(os.getenv('NANOGPT_LEVEL0_NOTEBOOK_SEED', '1337'))
RUN = ROOT / f'{OPT}_seed_{SEED}'

manifest = json.loads((RUN / 'manifest.json').read_text())
completion = json.loads((RUN / 'run_complete.json').read_text())
selected = json.loads((RUN / 'selected_checkpoint_metrics.json').read_text())
df = pd.read_csv(RUN / 'metrics.csv')
print('run:', RUN)
print('parameters:', f"{manifest['parameter_count']:,}")
print('device:', manifest['device'])
print('completed steps:', completion['optimizer_steps'])
print('selected checkpoint:', selected)
df.tail()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df.step, df.train_loss, label='train')
ax.plot(df.step, df.val_loss, label='validation')
final_test = df.dropna(subset=['test_loss'])
if len(final_test):
    ax.scatter(final_test.step, final_test.test_loss, s=70, marker='*', label='final test')
ax.scatter([selected['selected_step']], [selected['validation_loss']], s=60, marker='D', label='selected validation')
ax.set(xlabel='optimizer step', ylabel='cross-entropy loss', title=f'{OPT} seed {SEED}: loss')
ax.grid(alpha=.25); ax.legend(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df.step, df.train_perplexity, label='train')
ax.plot(df.step, df.val_perplexity, label='validation')
final_test = df.dropna(subset=['test_perplexity'])
if len(final_test):
    ax.scatter(final_test.step, final_test.test_perplexity, s=70, marker='*', label='final test')
ax.set(xlabel='optimizer step', ylabel='perplexity', title=f'{OPT} seed {SEED}: perplexity')
ax.grid(alpha=.25); ax.legend(); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for split in ['train', 'val']:
    ax.plot(df.step, 100 * df[f'{split}_accuracy'], label=split)
final_test = df.dropna(subset=['test_accuracy'])
if len(final_test):
    ax.scatter(final_test.step, 100 * final_test.test_accuracy, s=70, marker='*', label='final test')
ax.set(xlabel='optimizer step', ylabel='exact next-BPE-token accuracy (%)', title=f'{OPT} seed {SEED}: secondary accuracy metric')
ax.grid(alpha=.25); ax.legend(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(df.step, df.learning_rate)
axes[0].set(xlabel='optimizer step', ylabel='learning rate', title='Warmup + cosine schedule')
axes[0].grid(alpha=.25)
axes[1].plot(df.step, df.val_generalization_gap, label='validation - train')
axes[1].axhline(0, linewidth=1)
axes[1].set(xlabel='optimizer step', ylabel='loss gap', title='Generalization gap')
axes[1].grid(alpha=.25); axes[1].legend()
plt.show()


In [ ]:
files = sorted(RUN.glob('weightwatcher_step_*.csv'))
if not files:
    raise FileNotFoundError('No WeightWatcher files found')
ww = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
ww['alpha'] = pd.to_numeric(ww['alpha'], errors='coerce')
ww = ww[np.isfinite(ww.alpha)].copy()
summary = ww.groupby(['step', 'matrix_type'], as_index=False).alpha.mean()
fig, ax = plt.subplots(figsize=(11, 6))
for matrix_type, group in summary.groupby('matrix_type'):
    ax.plot(group.step, group.alpha, marker='o', label=matrix_type)
ax.axhline(2.0, linestyle='--', linewidth=1, label='alpha = 2')
ax.set(xlabel='optimizer step', ylabel='mean WeightWatcher alpha across blocks', title=f'{OPT} seed {SEED}: spectral trajectory')
ax.grid(alpha=.25); ax.legend(ncol=2); plt.show()
ww[['step','matrix_name','matrix_type','block','alpha','D','xmin','num_evals']].tail(24)


In [ ]:
final = df.iloc[-1]
print(f"best validation loss: {selected['validation_loss']:.6f} at step {selected['selected_step']}")
print(f"selected-checkpoint test loss: {selected['test_loss']:.6f}")
print(f"selected-checkpoint test perplexity: {selected['test_perplexity']:.3f}")
print(f"final validation loss: {final.val_loss:.6f}")
print(f"final test loss: {final.test_loss:.6f}")
